# Biohub - Cell Tracking During Development
## Score: 0.898

## Configuration

In [ ]:
import os
from pathlib import Path

COMP_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
TEST_DIR = f"{COMP_DIR}/test"
ARTIFACTS_ROOT = Path(
    "/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts"
)
WEIGHTS_CANDIDATES = [
    Path("/kaggle/input/datasets/hongdaekim/biohub-350ep-checkpoint-pin-v1"),
    Path("/kaggle/input/hongdaekim/biohub-350ep-checkpoint-pin-v1"),
    Path("/kaggle/input/biohub-350ep-checkpoint-pin-v1"),
    Path(
        "/kaggle/input/datasets/hongdaekim/"
        "biohub-350ep-checkpoint-pin-v1/biohub-350ep-checkpoint-pin-v1"
    ),
]
REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0
MOTION_RELINK_TIGHT_UM = 6.0
MOTION_RELINK_RELAXED_UM = 10.0
MOTION_RELINK_VELOCITY_WEIGHT = 0.5
MOTION_RELINK_LEARNED_BONUS = 1.0
MOTION_RELINK_MAX_FRAME_NODES = 2600
GAP_CLOSE_UM = 6.0
GAP_CLOSE_REUSE_UM = 3.2
GAP_CLOSE_MAX_ADDED_FRAC = 0.05
GAP_CLOSE_MAX_ADDED_ABS = 2000
GAP2_MAX_TOTAL_UM = 10.2
GAP2_MAX_STEP_UM = 4.4
GAP2_MAX_LINKS_FRAC = 0.0045
GAP2_MAX_LINKS_ABS = 180
GAP2_FRAME_FRAC_CAP = 0.006
SAFE_DIV_MAX_UM = 4.66
SAFE_DIV_SISTER_MAX_UM = 8.5
SAFE_DIV_EXISTING_CHILD_MAX_UM = 7.65
SAFE_DIV_FRAME_FRAC_CAP = 0.0076
SAFE_DIV_GLOBAL_FRAC_CAP = 0.00375
LINEFIT_WEIGHT = 0.8
LINEFIT_WINDOW = 2
MIN_TRACK_LEN = 6
KEEP_DIVISION_COMPONENTS = True
ADAPTIVE_SHORT_TRACK_RESCUE = True
SHORT_TRACK_RESCUE_MIN_LEN = 3
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = 0.05
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = 0.02
SHORT_TRACK_RESCUE_MAX_NODES_ABS = 400
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = 0.35
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = 6.0
OUTPUT_PATH = "/kaggle/working/submission.csv"

if (ARTIFACTS_ROOT / "repo").exists():
    ARTIFACTS = ARTIFACTS_ROOT
elif (ARTIFACTS_ROOT / "cellmot-baseline-artifacts" / "repo").exists():
    ARTIFACTS = ARTIFACTS_ROOT / "cellmot-baseline-artifacts"
else:
    ARTIFACTS = ARTIFACTS_ROOT

WEIGHTS_SRC = next(
    (
        path
        for path in WEIGHTS_CANDIDATES
        if (path / "edge_predictor_best.pth").exists()
    ),
    None,
)
if WEIGHTS_SRC is None:
    for path in Path("/kaggle/input").rglob("edge_predictor_best.pth"):
        if "thibautgoldsborough" in str(path):
            continue
        WEIGHTS_SRC = path.parent
        break

COMP_DIR, ARTIFACTS, WEIGHTS_SRC, OUTPUT_PATH


## Offline Install

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

assert WEIGHTS_SRC is not None, (
    "350ep checkpoint not found under /kaggle/input. "
    "Attach hongdaekim/biohub-350ep-checkpoint-pin-v1 and re-run."
)
assert (ARTIFACTS / "wheels").exists(), f"Missing wheels at {ARTIFACTS / 'wheels'}"
assert (ARTIFACTS / "repo").exists(), f"Missing repo at {ARTIFACTS / 'repo'}"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        str(ARTIFACTS / "wheels"),
        "--upgrade-strategy",
        "only-if-needed",
        "tracksdata",
        "zarr>=3.0.10",
        "pyscipopt",
    ],
    check=True,
)

shutil.copytree(ARTIFACTS / "repo", REPO_DIR, dirs_exist_ok=True)
if (ARTIFACTS / "weights").exists():
    shutil.copytree(ARTIFACTS / "weights", Path(REPO_DIR) / "weights", dirs_exist_ok=True)

weights_dir = Path(REPO_DIR) / "weights" / METHOD / "split_0"
weights_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(WEIGHTS_SRC / "edge_predictor_best.pth", weights_dir / "edge_predictor_best.pth")
config_src = WEIGHTS_SRC / "config.json"
if config_src.exists():
    shutil.copy2(config_src, weights_dir / "config.json")

sys.path.insert(0, f"{REPO_DIR}/src")
WEIGHTS_SRC, sorted(weights_dir.iterdir()), (weights_dir / "edge_predictor_best.pth").stat().st_size


## Test Split

In [ ]:
import json
from pathlib import Path

test_stems = sorted(
    path.name.replace(".zarr", "")
    for path in Path(TEST_DIR).glob("*.zarr")
)
splits_path = Path(REPO_DIR) / "kaggle_test_splits.json"
splits_path.write_text(
    json.dumps([{"split": 0, "train": [], "test": test_stems}])
)
len(test_stems), test_stems[:5]

## Detection TTA

In [ ]:
from pathlib import Path

predict_script = Path(REPO_DIR) / "scripts" / "predict_unet_transformer.py"
source = predict_script.read_text()

FOUR_WAY_TTA = '''        if cfg.det_tta:
            tta_flips = [(-1,), (-2,), (-2, -1)]
            for dims in tta_flips:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
            for f in range(W):
                det_logits[f] = det_logits[f] / 4'''

EIGHT_WAY_TTA = '''        if cfg.det_tta:
            views = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
                views += 1
            for k in (1, 3):
                imgs_rot = torch.rot90(imgs, k, dims=(-2, -1))
                _, det_rot = model.encode(imgs_rot)
                for f in range(W):
                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -k, dims=(-2, -1))
                del imgs_rot, det_rot
                views += 1
            imgs_t = imgs.transpose(-1, -2)
            _, det_t = model.encode(imgs_t)
            for f in range(W):
                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)
            del imgs_t, det_t
            views += 1
            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
            _, det_at = model.encode(imgs_at)
            for f in range(W):
                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))
            del imgs_at, det_at
            views += 1
            for f in range(W):
                det_logits[f] = det_logits[f] / views'''

DET_TTA_PATCHED = FOUR_WAY_TTA in source
if DET_TTA_PATCHED:
    source = source.replace(FOUR_WAY_TTA, EIGHT_WAY_TTA)

DET_TTA_FLAG = "--det-tta" in source
DET_TTA_FORCED = False
if not DET_TTA_FLAG:
    for old, new in (
        ("det_tta: bool = False", "det_tta: bool = True"),
        ("det_tta = False", "det_tta = True"),
        ('"det_tta": False', '"det_tta": True'),
    ):
        if old in source:
            source = source.replace(old, new)
            DET_TTA_FORCED = True

predict_script.write_text(source)

if not DET_TTA_PATCHED:
    index = source.find("det_tta")
    print(source[max(0, index - 1000):index + 1500] if index >= 0 else "det_tta absent")

DET_TTA_PATCHED, DET_TTA_FLAG, DET_TTA_FORCED

## Inference

In [ ]:
import os
import subprocess

cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    TEST_DIR,
    "--splits",
    "kaggle_test_splits.json",
    "--split",
    "0",
    "--weights",
    WEIGHTS,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    cmd.append("--use-ilp")
if DET_TTA_FLAG:
    cmd.append("--det-tta")

print(" ".join(cmd))
subprocess.run(
    cmd,
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": "src"},
    check=True,
)

## Graph Repair


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

pred_dirs = sorted(Path(REPO_DIR, "predictions").glob(f"*/{METHOD}/split_0"))
assert pred_dirs, "No prediction directory found"
PRED_DIR = pred_dirs[0]

repair_code = r"""
import csv
import math
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import zarr
from scipy.optimize import linear_sum_assignment

pred_dir = Path(sys.argv[1])
out_csv = Path(sys.argv[2])
args = sys.argv[3:]
(
    tight_um,
    relaxed_um,
    velocity_weight,
    learned_bonus,
    max_frame_nodes,
    gap_close_um,
    gap_close_reuse_um,
    gap_close_max_added_frac,
    gap_close_max_added_abs,
    gap2_max_total_um,
    gap2_max_step_um,
    gap2_max_links_frac,
    gap2_max_links_abs,
    gap2_frame_frac_cap,
    safe_div_max_um,
    safe_div_sister_max_um,
    safe_div_existing_child_max_um,
    safe_div_frame_frac_cap,
    safe_div_global_frac_cap,
    linefit_weight,
    linefit_window,
    min_track_len,
    keep_division_components,
    adaptive_rescue,
    rescue_min_len,
    rescue_trigger_frac,
    rescue_max_frac,
    rescue_max_abs,
    rescue_min_prob,
    rescue_max_dist,
) = (
    float(args[0]),
    float(args[1]),
    float(args[2]),
    float(args[3]),
    int(args[4]),
    float(args[5]),
    float(args[6]),
    float(args[7]),
    int(args[8]),
    float(args[9]),
    float(args[10]),
    float(args[11]),
    int(args[12]),
    float(args[13]),
    float(args[14]),
    float(args[15]),
    float(args[16]),
    float(args[17]),
    float(args[18]),
    float(args[19]),
    int(args[20]),
    int(args[21]),
    args[22] == "1",
    args[23] == "1",
    int(args[24]),
    float(args[25]),
    float(args[26]),
    int(args[27]),
    float(args[28]),
    float(args[29]),
)
scale = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)


def load_graph(path: Path):
    root = zarr.open(str(path), mode="r")
    node_ids = np.asarray(root["nodes"]["ids"][:])
    t = np.asarray(root["nodes"]["props"]["t"]["values"][:])
    z = np.asarray(root["nodes"]["props"]["z"]["values"][:])
    y = np.asarray(root["nodes"]["props"]["y"]["values"][:])
    x = np.asarray(root["nodes"]["props"]["x"]["values"][:])
    if "solution" in root["nodes"]["props"]:
        keep = np.asarray(root["nodes"]["props"]["solution"]["values"][:]).astype(bool)
    else:
        keep = np.ones(len(node_ids), dtype=bool)
    nodes = {}
    for i, node_id in enumerate(node_ids):
        if not keep[i]:
            continue
        nodes[int(node_id)] = {
            "node_id": int(node_id),
            "t": int(t[i]),
            "z": float(z[i]),
            "y": float(y[i]),
            "x": float(x[i]),
        }
    edges_raw = np.asarray(root["edges"]["ids"][:])
    if "solution" in root["edges"]["props"]:
        edge_keep = np.asarray(root["edges"]["props"]["solution"]["values"][:]).astype(bool)
    else:
        edge_keep = np.ones(len(edges_raw), dtype=bool)
    props = root["edges"]["props"]
    prob_arr = None
    for key in ("edge_prob", "score", "prob", "weight"):
        if key in props:
            prob_arr = np.asarray(props[key]["values"][:])
            break
    edges = []
    edge_probs = {}
    for i, (source, target) in enumerate(edges_raw):
        if not edge_keep[i]:
            continue
        source, target = int(source), int(target)
        if source not in nodes or target not in nodes:
            continue
        prob = float(prob_arr[i]) if prob_arr is not None else 0.0
        edges.append({"source_id": source, "target_id": target, "edge_prob": prob})
        edge_probs[(source, target)] = max(edge_probs.get((source, target), float("-inf")), prob)
    return nodes, edges, edge_probs


def position_um(node):
    return np.array(
        [node["z"] * scale[0], node["y"] * scale[1], node["x"] * scale[2]],
        dtype=np.float64,
    )


def edge_distance_um(source, target):
    return float(np.linalg.norm(position_um(source) - position_um(target)))


def next_node_id(nodes):
    return (max(nodes) + 1) if nodes else 1


def learned_prob(edge_probs, source_id, target_id):
    value = edge_probs.get((source_id, target_id), 0.0)
    try:
        value = float(value)
    except (TypeError, ValueError):
        return 0.0
    if not np.isfinite(value):
        return 0.0
    if value < 0.0 or value > 1.0:
        value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
    return float(np.clip(value, 0.0, 1.0))


def single_successor_map(edges):
    by_source = defaultdict(list)
    for edge in edges:
        by_source[int(edge["source_id"])].append(int(edge["target_id"]))
    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}


def single_predecessor_map(edges):
    by_target = defaultdict(list)
    for edge in edges:
        by_target[int(edge["target_id"])].append(int(edge["source_id"]))
    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}


def motion_replace(nodes, edges, edge_probs, stats):
    if not nodes:
        return []
    ids_by_t = defaultdict(list)
    for node_id, node in nodes.items():
        ids_by_t[int(node["t"])].append(node_id)
    for ids in ids_by_t.values():
        ids.sort()
    if ids_by_t and max(len(v) for v in ids_by_t.values()) > max_frame_nodes:
        stats["motion_relink_skipped_large_frame"] = 1
        return list(edges)

    learned = {}
    for edge in edges:
        try:
            prob = float(edge.get("edge_prob", 0.0))
        except (TypeError, ValueError):
            continue
        if np.isfinite(prob):
            key = (int(edge["source_id"]), int(edge["target_id"]))
            learned[key] = max(learned.get(key, float("-inf")), prob)
    for key, prob in edge_probs.items():
        learned[key] = max(learned.get(key, float("-inf")), float(prob))

    pos = {node_id: position_um(node) for node_id, node in nodes.items()}
    predecessor_pos = {}
    selected = []

    def assign_pass(source_ids, target_ids, gate_um):
        if not source_ids or not target_ids:
            return []
        big = gate_um * 1000.0 + 1.0
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        for i, source_id in enumerate(source_ids):
            source_pos = pos[source_id]
            prev = predecessor_pos.get(source_id)
            predicted = (
                source_pos
                if prev is None
                else source_pos + velocity_weight * (source_pos - prev)
            )
            for j, target_id in enumerate(target_ids):
                target_pos = pos[target_id]
                raw = float(np.linalg.norm(target_pos - source_pos))
                if raw > gate_um:
                    continue
                motion = float(np.linalg.norm(target_pos - predicted))
                prob = learned_prob(learned, source_id, target_id)
                raw_dist[i, j] = raw
                prob_matrix[i, j] = prob
                cost[i, j] = motion + 0.05 * raw - learned_bonus * prob
        rows, cols = linear_sum_assignment(cost)
        matches = []
        for r, c in zip(rows, cols):
            if cost[r, c] >= big:
                continue
            matches.append(
                (
                    source_ids[int(r)],
                    target_ids[int(c)],
                    float(raw_dist[r, c]),
                    float(prob_matrix[r, c]),
                )
            )
        return matches

    for t in sorted(ids_by_t):
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue
        unmatched_s = set(source_ids)
        unmatched_t = set(target_ids)
        frame_matches = []
        for pass_name, gate in (("tight", tight_um), ("relaxed", relaxed_um)):
            s_ids = [i for i in source_ids if i in unmatched_s]
            t_ids = [i for i in target_ids if i in unmatched_t]
            for source_id, target_id, raw, prob in assign_pass(s_ids, t_ids, gate):
                if source_id not in unmatched_s or target_id not in unmatched_t:
                    continue
                unmatched_s.remove(source_id)
                unmatched_t.remove(target_id)
                frame_matches.append((source_id, target_id, raw, prob, pass_name))
                stats[f"motion_relink_{pass_name}_edges"] += 1
        for source_id, target_id, raw, prob, pass_name in frame_matches:
            selected.append(
                {
                    "source_id": source_id,
                    "target_id": target_id,
                    "edge_prob": prob,
                    "distance_um": raw,
                }
            )
            predecessor_pos[target_id] = pos[source_id]
        stats["motion_relink_frames"] += 1

    if not selected:
        stats["motion_relink_fallback_raw"] = 1
        fallback = []
        for edge in edges:
            item = dict(edge)
            item["distance_um"] = float(
                np.linalg.norm(pos[edge["source_id"]] - pos[edge["target_id"]])
            )
            fallback.append(item)
        return fallback
    stats["motion_relink_replaced_raw_edges"] = len(edges)
    stats["motion_relink_edges"] = len(selected)
    return selected


def close_single_frame_gaps(nodes, edges, stats):
    if not edges:
        return nodes, edges
    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming

    ends_by_t = defaultdict(list)
    starts_by_t = defaultdict(list)
    isolated_by_t = defaultdict(list)
    for node_id, node in nodes.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t[t].append(node_id)
        if node_id not in incoming:
            starts_by_t[t].append(node_id)
        if node_id not in incident:
            isolated_by_t[t].append(node_id)

    max_synthetic = min(
        gap_close_max_added_abs,
        max(1, int(round(len(nodes) * gap_close_max_added_frac)))
        if gap_close_max_added_frac > 0
        else 0,
    )
    nid = next_node_id(nodes)
    used_starts = set()
    used_isolated = set()
    synthetic_added = 0
    new_edges = []
    threshold_um = gap_close_um * 2.0

    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = [sid for sid in starts_by_t.get(t + 2, []) if sid not in used_starts]
        if not end_ids or not start_ids:
            continue
        end_points = [position_um(nodes[eid]) for eid in end_ids]
        start_points = [position_um(nodes[sid]) for sid in start_ids]
        d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)
        for i, ep in enumerate(end_points):
            for j, sp in enumerate(start_points):
                d[i, j] = float(np.linalg.norm(ep - sp))
        stats["gap_candidates"] += int((d <= threshold_um).sum())
        big = threshold_um * 1000.0 + 1.0
        cost = np.where(d <= threshold_um, d, big)
        rows, cols = linear_sum_assignment(cost)
        for r, c in zip(rows, cols):
            if d[r, c] > threshold_um:
                continue
            source_id = end_ids[int(r)]
            target_id = start_ids[int(c)]
            if source_id in outgoing or target_id in used_starts:
                continue
            source = nodes[source_id]
            target = nodes[target_id]
            mid_t = int(source["t"]) + 1
            mid_point = {
                "z": (float(source["z"]) + float(target["z"])) / 2.0,
                "y": (float(source["y"]) + float(target["y"])) / 2.0,
                "x": (float(source["x"]) + float(target["x"])) / 2.0,
            }
            middle_id = None
            candidates = [i for i in isolated_by_t.get(mid_t, []) if i not in used_isolated]
            if candidates:
                mid_um = position_um(mid_point)
                distances = [float(np.linalg.norm(position_um(nodes[i]) - mid_um)) for i in candidates]
                best_idx = int(np.argmin(distances))
                if distances[best_idx] <= gap_close_reuse_um:
                    middle_id = candidates[best_idx]
                    used_isolated.add(middle_id)
                    stats["gap_reused_existing"] += 1
            if middle_id is None:
                if synthetic_added >= max_synthetic:
                    stats["gap_skipped_node_cap"] += 1
                    continue
                middle_id = nid
                nid += 1
                nodes[middle_id] = {
                    "node_id": middle_id,
                    "t": mid_t,
                    "z": mid_point["z"],
                    "y": mid_point["y"],
                    "x": mid_point["x"],
                }
                synthetic_added += 1
                stats["gap_inserted_synthetic"] += 1
            middle = nodes[middle_id]
            new_edges.append(
                {
                    "source_id": source_id,
                    "target_id": middle_id,
                    "edge_prob": 0.0,
                    "distance_um": edge_distance_um(source, middle),
                }
            )
            new_edges.append(
                {
                    "source_id": middle_id,
                    "target_id": target_id,
                    "edge_prob": 0.0,
                    "distance_um": edge_distance_um(middle, target),
                }
            )
            outgoing.add(source_id)
            incoming.add(middle_id)
            outgoing.add(middle_id)
            incoming.add(target_id)
            used_starts.add(target_id)
            stats["gap_pairs_selected"] += 1
            stats["gap_added_edges"] += 2
    stats["gap_added_nodes"] = synthetic_added
    if new_edges:
        edges = [*edges, *new_edges]
    return nodes, edges


def recover_strict_gap2(nodes, edges, stats):
    if not edges or not nodes:
        return nodes, edges
    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    predecessor = single_predecessor_map(edges)
    successor = single_successor_map(edges)
    ends_by_t = defaultdict(list)
    starts_by_t = defaultdict(list)
    for node_id, node in nodes.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t[t].append(node_id)
        if node_id not in incoming:
            starts_by_t[t].append(node_id)

    cap = min(gap2_max_links_abs, max(1, int(round(len(edges) * gap2_max_links_frac))))
    proposals = []
    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = starts_by_t.get(t + 3, [])
        if not end_ids or not start_ids:
            continue
        for end_id in end_ids:
            end_pos = position_um(nodes[end_id])
            for start_id in start_ids:
                start_pos = position_um(nodes[start_id])
                dist = float(np.linalg.norm(start_pos - end_pos))
                if dist > gap2_max_total_um or dist / 3.0 > gap2_max_step_um:
                    continue
                step = (start_pos - end_pos) / 3.0
                context_penalty = 0.0
                ok_context = False
                prev_id = predecessor.get(end_id)
                if prev_id is not None:
                    prev_step = end_pos - position_um(nodes[prev_id])
                    prev_norm = float(np.linalg.norm(prev_step))
                    step_norm = float(np.linalg.norm(step))
                    if prev_norm <= 0.01 or step_norm <= 0.01:
                        ok_context = True
                    else:
                        cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))
                        if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:
                            ok_context = True
                        context_penalty += max(0.0, 0.25 - cos)
                next_id = successor.get(start_id)
                if next_id is not None:
                    next_step = position_um(nodes[next_id]) - start_pos
                    next_norm = float(np.linalg.norm(next_step))
                    step_norm = float(np.linalg.norm(step))
                    if next_norm <= 0.01 or step_norm <= 0.01:
                        ok_context = True
                    else:
                        cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))
                        if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:
                            ok_context = True
                        context_penalty += max(0.0, 0.25 - cos)
                if not ok_context:
                    continue
                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))

    proposals.sort(key=lambda item: item[0])
    stats["gap2_candidates"] = len(proposals)
    if not proposals:
        return nodes, edges

    selected = []
    used_ends = set()
    used_starts = set()
    per_frame_count = {}
    for proposal in proposals:
        if len(selected) >= cap:
            stats["gap2_skipped_cap"] += 1
            break
        _, end_id, start_id, t, _ = proposal
        if end_id in used_ends or start_id in used_starts:
            continue
        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * gap2_frame_frac_cap)))
        if per_frame_count.get(t, 0) >= frame_cap:
            continue
        selected.append(proposal)
        used_ends.add(end_id)
        used_starts.add(start_id)
        per_frame_count[t] = per_frame_count.get(t, 0) + 1

    if not selected:
        return nodes, edges

    nid = next_node_id(nodes)
    new_edges = []
    for _, end_id, start_id, t, _ in selected:
        source = nodes[end_id]
        target = nodes[start_id]
        previous_id = end_id
        inserted = 0
        for k in (1, 2):
            frac = k / 3.0
            mid_t = int(source["t"]) + k
            midpoint = {
                "z": float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,
                "y": float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,
                "x": float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,
            }
            node_id = nid
            nid += 1
            nodes[node_id] = {
                "node_id": node_id,
                "t": mid_t,
                "z": midpoint["z"],
                "y": midpoint["y"],
                "x": midpoint["x"],
            }
            inserted += 1
            current = nodes[node_id]
            new_edges.append(
                {
                    "source_id": previous_id,
                    "target_id": node_id,
                    "edge_prob": 0.0,
                    "distance_um": edge_distance_um(nodes[previous_id], current),
                }
            )
            previous_id = node_id
        new_edges.append(
            {
                "source_id": previous_id,
                "target_id": start_id,
                "edge_prob": 0.0,
                "distance_um": edge_distance_um(nodes[previous_id], target),
            }
        )
        stats["gap2_pairs_selected"] += 1
        stats["gap2_added_nodes"] += inserted
        stats["gap2_added_edges"] += 3
    return nodes, [*edges, *new_edges]


def add_safe_divisions(nodes, edges, stats):
    if not edges or not nodes:
        return edges
    out_by_source = defaultdict(list)
    incoming = set()
    for edge in edges:
        out_by_source[int(edge["source_id"])].append(edge)
        incoming.add(int(edge["target_id"]))
    ids_by_t = defaultdict(list)
    for node_id, node in nodes.items():
        ids_by_t[int(node["t"])].append(node_id)
    existing_edges = {(int(e["source_id"]), int(e["target_id"])) for e in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * safe_div_global_frac_cap)))
    added = []
    used_targets = set()

    for t in sorted(ids_by_t):
        child_frame_ids = ids_by_t.get(t + 1, [])
        if not child_frame_ids:
            continue
        source_ids = [nid for nid in ids_by_t[t] if len(out_by_source.get(nid, [])) == 1]
        candidate_ids = [
            nid for nid in child_frame_ids if nid not in incoming and nid not in used_targets
        ]
        if not source_ids or not candidate_ids:
            continue
        frame_cap = max(1, int(round(len(source_ids) * safe_div_frame_frac_cap)))
        proposals = []
        for source_id in source_ids:
            source = nodes[source_id]
            existing_child_id = int(out_by_source[source_id][0]["target_id"])
            existing_child = nodes.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_dist = edge_distance_um(source, existing_child)
            if child_dist > safe_div_existing_child_max_um:
                continue
            for candidate_id in candidate_ids:
                if (source_id, candidate_id) in existing_edges:
                    continue
                candidate = nodes[candidate_id]
                parent_dist = edge_distance_um(source, candidate)
                if parent_dist > safe_div_max_um:
                    continue
                sister_dist = edge_distance_um(existing_child, candidate)
                if sister_dist > safe_div_sister_max_um:
                    continue
                score = parent_dist + 0.15 * sister_dist
                proposals.append((score, source_id, candidate_id, parent_dist))
        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])
        added_this_frame = 0
        for _, source_id, candidate_id, parent_dist in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            added.append(
                {
                    "source_id": source_id,
                    "target_id": candidate_id,
                    "edge_prob": 0.0,
                    "distance_um": parent_dist,
                }
            )
            used_targets.add(candidate_id)
            incoming.add(candidate_id)
            added_this_frame += 1
    if added:
        stats["safe_divisions_added"] = len(added)
        return [*edges, *added]
    return edges


def linefit_smooth(nodes, edges, stats):
    if linefit_weight <= 0 or linefit_window <= 0 or not edges:
        return nodes
    predecessor = defaultdict(list)
    successor = defaultdict(list)
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes.get(source_id)
        target = nodes.get(target_id)
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        successor[source_id].append(target_id)
        predecessor[target_id].append(source_id)

    original_pos = {
        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)
        for node_id, node in nodes.items()
    }
    updated_pos = {}
    weight = float(np.clip(linefit_weight, 0.0, 1.0))

    for node_id in sorted(nodes):
        neighbourhood = [(0, node_id)]
        current = node_id
        for step in range(1, linefit_window + 1):
            prev_ids = predecessor.get(current, [])
            if len(prev_ids) != 1:
                break
            current = prev_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((-step, current))
        current = node_id
        for step in range(1, linefit_window + 1):
            next_ids = successor.get(current, [])
            if len(next_ids) != 1:
                break
            current = next_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((step, current))
        if len(neighbourhood) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue
        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)
        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])
        fitted = np.array(
            [np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)],
            dtype=np.float64,
        )
        if not np.isfinite(fitted).all():
            stats["linefit_skipped_nodes"] += 1
            continue
        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted

    for node_id, pos in updated_pos.items():
        nodes[node_id]["z"] = float(pos[0])
        nodes[node_id]["y"] = float(pos[1])
        nodes[node_id]["x"] = float(pos[2])
    stats["linefit_smoothed_nodes"] = len(updated_pos)
    return nodes


def filter_short_tracks(nodes, edges, stats):
    if min_track_len <= 1 or not edges:
        return nodes, edges
    parent = {node_id: node_id for node_id in nodes}

    def find(node_id):
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a, b):
        if a not in parent or b not in parent:
            return
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    out_count = defaultdict(int)
    for edge in edges:
        s, t = int(edge["source_id"]), int(edge["target_id"])
        union(s, t)
        out_count[s] += 1
    components = defaultdict(list)
    for node_id in nodes:
        components[find(node_id)].append(node_id)
    component_edges = defaultdict(list)
    for edge in edges:
        s, t = int(edge["source_id"]), int(edge["target_id"])
        if s in parent and t in parent:
            component_edges[find(s)].append(edge)
    keep = set()
    for root, members in components.items():
        has_div = any(out_count[n] >= 2 for n in members)
        if len(members) >= min_track_len or (keep_division_components and has_div):
            keep.update(members)
    if not keep:
        stats["short_track_filter_skipped_all"] = 1
        return nodes, edges
    removed_before = len(nodes) - len(keep)
    if removed_before <= 0:
        return nodes, edges
    if adaptive_rescue:
        removed_frac = removed_before / max(len(nodes), 1)
        if removed_frac >= rescue_trigger_frac:
            budget = min(rescue_max_abs, max(0, int(round(len(nodes) * rescue_max_frac))))
            stats["short_track_rescue_triggered"] = 1
            stats["short_track_rescue_budget"] = budget
            proposals = []
            for root, members in components.items():
                if set(members) & keep:
                    continue
                if len(members) < rescue_min_len or len(members) >= min_track_len:
                    continue
                c_edges = component_edges.get(root, [])
                if not c_edges:
                    continue
                probs = []
                dists = []
                for edge in c_edges:
                    try:
                        probs.append(float(edge.get("edge_prob", 0.0)))
                    except (TypeError, ValueError):
                        pass
                    try:
                        dists.append(float(edge.get("distance_um", np.nan)))
                    except (TypeError, ValueError):
                        pass
                mean_prob = float(np.mean(probs)) if probs else 0.0
                finite_dists = [d for d in dists if np.isfinite(d)]
                mean_dist = float(np.mean(finite_dists)) if finite_dists else float("inf")
                if mean_prob < rescue_min_prob or mean_dist > rescue_max_dist:
                    continue
                score = mean_prob - 0.02 * mean_dist + 0.004 * len(members)
                proposals.append((score, len(members), members))
            proposals.sort(reverse=True)
            rescued_nodes = 0
            rescued_components = 0
            for _, size, members in proposals:
                if budget <= 0 or rescued_nodes + size > budget:
                    continue
                keep.update(members)
                rescued_nodes += size
                rescued_components += 1
            stats["short_track_rescue_components"] = rescued_components
            stats["short_track_rescue_nodes"] = rescued_nodes
    kept_nodes = {i: n for i, n in nodes.items() if i in keep}
    kept_edges = [
        e
        for e in edges
        if int(e["source_id"]) in kept_nodes and int(e["target_id"]) in kept_nodes
    ]
    stats["short_track_nodes_removed"] = len(nodes) - len(kept_nodes)
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


rows = []
for geff in sorted(pred_dir.glob("*.geff")):
    nodes, raw_edges, edge_probs = load_graph(geff)
    stats = defaultdict(int)
    edges = motion_replace(nodes, raw_edges, edge_probs, stats)
    nodes, edges = close_single_frame_gaps(nodes, edges, stats)
    nodes, edges = recover_strict_gap2(nodes, edges, stats)
    edges = add_safe_divisions(nodes, edges, stats)
    incident = {int(e["source_id"]) for e in edges} | {int(e["target_id"]) for e in edges}
    if incident:
        pruned = len(nodes) - len(incident)
        nodes = {i: n for i, n in nodes.items() if i in incident}
        edges = [
            e
            for e in edges
            if int(e["source_id"]) in nodes and int(e["target_id"]) in nodes
        ]
        stats["pruned_isolated_nodes"] = pruned
    nodes, edges = filter_short_tracks(nodes, edges, stats)
    nodes = linefit_smooth(nodes, edges, stats)
    dataset = geff.stem
    for node_id, node in sorted(nodes.items()):
        rows.append(
            [
                dataset,
                "node",
                node_id,
                int(node["t"]),
                int(round(node["z"])),
                int(round(node["y"])),
                int(round(node["x"])),
                -1,
                -1,
            ]
        )
    for edge in edges:
        rows.append(
            [
                dataset,
                "edge",
                -1,
                -1,
                -1,
                -1,
                -1,
                int(edge["source_id"]),
                int(edge["target_id"]),
            ]
        )
    print(dataset, dict(stats), "nodes", len(nodes), "edges", len(edges))

with out_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(
        [
            "id",
            "dataset",
            "row_type",
            "node_id",
            "t",
            "z",
            "y",
            "x",
            "source_id",
            "target_id",
        ]
    )
    for index, row in enumerate(rows):
        writer.writerow([index, *row])
print("rows", len(rows), "out", out_csv)
"""

cmd = [
    sys.executable,
    "-c",
    repair_code,
    str(PRED_DIR),
    OUTPUT_PATH,
    str(MOTION_RELINK_TIGHT_UM),
    str(MOTION_RELINK_RELAXED_UM),
    str(MOTION_RELINK_VELOCITY_WEIGHT),
    str(MOTION_RELINK_LEARNED_BONUS),
    str(MOTION_RELINK_MAX_FRAME_NODES),
    str(GAP_CLOSE_UM),
    str(GAP_CLOSE_REUSE_UM),
    str(GAP_CLOSE_MAX_ADDED_FRAC),
    str(GAP_CLOSE_MAX_ADDED_ABS),
    str(GAP2_MAX_TOTAL_UM),
    str(GAP2_MAX_STEP_UM),
    str(GAP2_MAX_LINKS_FRAC),
    str(GAP2_MAX_LINKS_ABS),
    str(GAP2_FRAME_FRAC_CAP),
    str(SAFE_DIV_MAX_UM),
    str(SAFE_DIV_SISTER_MAX_UM),
    str(SAFE_DIV_EXISTING_CHILD_MAX_UM),
    str(SAFE_DIV_FRAME_FRAC_CAP),
    str(SAFE_DIV_GLOBAL_FRAC_CAP),
    str(LINEFIT_WEIGHT),
    str(LINEFIT_WINDOW),
    str(MIN_TRACK_LEN),
    "1" if KEEP_DIVISION_COMPONENTS else "0",
    "1" if ADAPTIVE_SHORT_TRACK_RESCUE else "0",
    str(SHORT_TRACK_RESCUE_MIN_LEN),
    str(SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC),
    str(SHORT_TRACK_RESCUE_MAX_NODES_FRAC),
    str(SHORT_TRACK_RESCUE_MAX_NODES_ABS),
    str(SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB),
    str(SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM),
]
print("repair", PRED_DIR, "->", OUTPUT_PATH)
subprocess.run(cmd, check=True)
Path(OUTPUT_PATH).exists(), Path(OUTPUT_PATH).stat().st_size


## Submission

In [ ]:
from pathlib import Path

assert Path(OUTPUT_PATH).exists(), f"Missing {OUTPUT_PATH}"
Path(OUTPUT_PATH).exists(), Path(OUTPUT_PATH).stat().st_size


## Submission Checks

In [ ]:
import csv
from pathlib import Path

expected = set(test_stems)
datasets = set()
row_count = 0
with Path(OUTPUT_PATH).open(encoding="utf-8") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames[0] == "id"
    for expected_id, row in enumerate(reader):
        assert int(row["id"]) == expected_id
        datasets.add(row["dataset"])
        row_count += 1

assert datasets == expected
assert row_count > 0
row_count, Path(OUTPUT_PATH).stat().st_size